In [ ]:
from src.load_params import load_params, format_duration
from src.scan_fcts import *
from src.astrometry_fcts import *
from src.hdf5_fcts import *
from astropy.io import fits 
import os
from astropy.wcs import WCS
from gen_timestreams import gen_tod
import matplotlib.patches as mpatches
from matplotlib.pyplot import cm
from astropy.visualization.wcsaxes import WCSAxes
import matplotlib.patches as patches
import matplotlib
import pickle
from matplotlib.path import Path

matplotlib.rcParams.update({'font.size': 11})
matplotlib.rcParams.update({'axes.grid':False})
matplotlib.rcParams.update({'xtick.direction':'out'})
matplotlib.rcParams.update({'ytick.direction':'out'})
matplotlib.rcParams.update({'xtick.top':True})
matplotlib.rcParams.update({'ytick.right':True})
matplotlib.rcParams.update({'legend.fontsize':'small'})
matplotlib.rcParams.update({'legend.frameon':False})
matplotlib.rcParams.update({'legend.fancybox':False})
matplotlib.rcParams.update({'legend.framealpha':0.6})
matplotlib.rcParams.update({'lines.dashed_pattern':[7,3]})
matplotlib.rcParams.update({'lines.dashed_pattern':[7,3]})

#Load the coordinates of the field using Astropy.
name='Goods-S Field'
c=SkyCoord.from_name(name)
ra = 0
rafield= c.ra.value
dec = c.dec.value
x_cen = ra
y_cen = dec
simu_sky_path = '/Users/thachdang/TIM_analysis/timestream_maker/fits_and_hdf5/pySIDES_from_uchuu_tile_0_1.414deg_x_1.414deg_fir_lines_res20arcsec_dnu4.0GHz_full_de_Looze_smoothed_MJy_sr.fits'
lat = -77.83 #deg
res = (1.22 * 240*10**(-6)/2) / np.pi * 180
    
hdr  = fits.getheader(simu_sky_path)
rafield = 53.11667
hdr['CRVAL1'] = ra
hdr['CRVAL2'] = dec
hdr['CRPIX1'] = 76
hdr['CRPIX2'] = 76
hdr['CDELT1'] = res
hdr['CDELT2'] = res
wcs = WCS(hdr, naxis=2) 

boxsize = 70

offset2 = 0.0145 #sky angle separation of detectors in the LW array
offset1 = 0.0182 #sky angle separation of detectors in the SW array
EL1, XEL1 = pixelOffset(51, offset1, 0)
separation=0.066
EL2, XEL2 = pixelOffset(64, offset2, 0)
dither_offset = 1/3 * res

In [ ]:
list_T_duration = np.linspace(1, 13, 13)  #hours
EL, XEL = EL1, XEL1
pixel_offsets = np.vstack((XEL, EL)).T

for i, T_duration in enumerate(list_T_duration):
    
    acquisition_frequency = 100 #Hz
    dt = 1/acquisition_frequency/3600*np.pi/3.14 #dt in hours. 


    az, alt, flag = genLocalPath_dither(az_size=1, alt_step=2.5*res, vertical_steps=7, N=0, acc=0.05, scan_v=0.1, dt=np.round(dt*3600, 3))
    
    # az0, alt0 = az, alt
    # az = np.tile(az, nrep)
    # alt = np.tile(alt, nrep)
    # flag = np.tile(flag, nrep)


    # Ttot = len(az)*dt
    # print(f'T={Ttot:.3f}h')
    # if( Ttot>13): continue # <--- skip unrealistic integration time to go faster
    # LST = np.linspace(-len(az)*dt/2, len(az)*dt/2, len(az), endpoint=False)
    # T = LST * 3600
    
    LST = np.arange(-T_duration/2,T_duration/2,dt) #hours
    T = LST*3600 #s
    pps = np.floor(T).astype(int)
    subsecond_ps = T-pps
    
    scan_path, scan_flag = genScanPath_dither(T, dt*3600, alt, az, flag, dither_offset=dither_offset)
    
    scan_path = scan_path[scan_flag==1]
    T= T[scan_flag==1]
    LST = LST[scan_flag==1]

    scan_path_sky, azel = genPointingPath(T, scan_path, LST, lat, dec, azel=True) 
    hitmap, xedges, yedges = np.histogram2d(scan_path_sky[:,0], scan_path_sky[:,1], bins=int(2/res))

    try: 
        az_unwrapped = (np.radians(azel[:, 0]) + np.pi) % (2 * np.pi) - np.pi
        coords = SkyCoord(alt=np.degrees(az_unwrapped)*u.deg,
                        az=azel[:, 1]*u.deg,
                        frame='altaz')
    except Exception as e: # <--- if unrealistic elevation, continue
        print(f"Warning: SkyCoord creation failed ({e}). Continuing...")
        print(f'break: time {T_duration}')
        continue

    #Generate the pointing on the sky of each detector
    pointing_paths = [genPointingPath(T, scan_path, LST, lat, dec, offsets) for offsets in pixel_offsets]

    #Generate the hitmap, using all the detectors. 
    xedges,yedges,hit_map = binMap(pointing_paths,res=res,dec=dec,ra=ra, shape = (151,151)) 

    depthmap = hit_map / hit_map.max()

    box_depthmap = depthmap[ hdr['CRPIX1']- boxsize//2:hdr['CRPIX1']+ boxsize//2,hdr['CRPIX2']- boxsize//2:hdr['CRPIX2']+ boxsize//2]
    peak_to_peak = np.max(box_depthmap) - np.min(box_depthmap)
    if(peak_to_peak > 0.5): continue  # <--- if the depth is too shallow, continue

    inds = np.nonzero(depthmap.flatten())
    bins = np.linspace(0,1,101)
    dist1, _ = np.histogram(depthmap.flatten()[inds], bins)
    dist1 = np.concatenate(([0],np.cumsum(dist1)))
    dist1 = dist1/np.max(dist1)
    dist2, _ = np.histogram(depthmap.flatten()[inds], bins, weights=depthmap.flatten()[inds])
    dist2 = np.concatenate(([0],np.cumsum(dist2)))
    dist2 = dist2/np.max(dist2)
    hits, time, stat_bins = dist1, dist2, bins

    fig = plt.figure(figsize=(6, 3), dpi=200)
    fig.suptitle(f'scan time: {T_duration} hours')
    # Left panel with WCS
    ax1 = fig.add_subplot(1, 2, 1, projection=wcs)
    # Right panel normal (no projection)
    ax2 = fig.add_subplot(1, 2, 2)

    # --- WCS axis formatting ---
    ax1.set_xlabel('RA [deg]')
    ax1.set_ylabel('Dec [deg]')
    ax1.coords[0].set_format_unit('deg', decimal=True)  # RA
    ax1.coords[1].set_format_unit('deg', decimal=True)  # Dec

    # --- WCS image ---
    ax1.imshow(depthmap, origin='lower', cmap='viridis')
    for level, color in zip((0.65,  1-peak_to_peak),('r','b')):
        ax1.contour(depthmap, levels=[level], colors=color, linewidths=1, alpha=0.5)
        ax2.plot((0, 1), level * np.asarray((1, 1)), c=color, alpha=0.7)
    ax2.plot(1 - time, stat_bins, c='k')

    ax1.set_title(f'ptp={peak_to_peak}')

    # Add the square
    rect = patches.Rectangle(
        (hdr['CRPIX1'] - boxsize//2,  hdr['CRPIX2'] - boxsize//2),          # bottom-left corner
        boxsize,          # width
        boxsize,          # height
        linewidth=1.5,
        edgecolor='magenta',
        facecolor='none',
        alpha=0.5

    )
    ax1.add_patch(rect)  
    # fig.savefig(f'plot/plot_hits_gitters{ngitters}_steps{nsteps}_stepsize{step_size*3600:.2f}arcsecs_repeted{T_duration}.png')
    plt.show()